# 🎯 02 — RFM Analysis

> **Objective**: Segment customers using Recency, Frequency, and Monetary (RFM) analysis — the gold standard for marketing analytics.

| Metric | Definition |
|---|---|
| **Recency (R)** | Days since last purchase |
| **Frequency (F)** | Total number of orders |
| **Monetary (M)** | Total revenue generated |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
COLORS = {'primary': '#003B73', 'secondary': '#0074B7', 'success': '#27AE60', 'alert': '#BF212F'}
print("Libraries loaded")

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('../../02_Dataset/cleaned_data/sales_clean.csv', parse_dates=['order_date'])
print(f"Dataset: {df.shape[0]:,} rows")

# Reference date = 1 day after the last order
reference_date = df['order_date'].max() + pd.Timedelta(days=1)
print(f"Reference Date: {reference_date.date()}")

## 2. Calculate RFM Metrics

In [ ]:
rfm = df.groupby('customer_id').agg(
    customer_name=('customer_name', 'first'),
    segment=('segment', 'first'),
    country=('country', 'first'),
    Recency=('order_date', lambda x: (reference_date - x.max()).days),
    Frequency=('order_id', 'nunique'),
    Monetary=('sales', 'sum')
).reset_index()

print(f"Unique Customers: {len(rfm):,}")
print(f"\nRFM Summary:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))
rfm.head(10)

## 3. Assign RFM Scores (1–5)

In [ ]:
# Score 1-5 using quantiles. Recency is inverted (lower = better = higher score)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("RFM Score Distribution:")
print(rfm[['R_Score', 'F_Score', 'M_Score', 'RFM_Total']].describe().round(2))

## 4. Customer Segmentation Rules

In [ ]:
def rfm_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f >= 1 and m >= 2:
        return 'Potential Loyalists'
    elif r == 3 and f == 3:
        return 'Need Attention'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost Customers'
    elif r <= 2 and f >= 2:
        return 'Hibernating'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)
segment_counts = rfm['Segment'].value_counts()
print("Customer Segments:")
print(segment_counts)

## 5. Visualize RFM Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, color, label in zip(axes,
    ['Recency', 'Frequency', 'Monetary'],
    [COLORS['alert'], COLORS['secondary'], COLORS['success']],
    ['Days', 'Orders', 'Revenue ($)']):
    sns.histplot(rfm[col], bins=40, kde=True, color=color, ax=ax, edgecolor='white')
    ax.axvline(rfm[col].mean(), color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'{col} Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel(label)
plt.suptitle('RFM Metric Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/rfm_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Segment treemap
seg_summary = rfm.groupby('Segment').agg(
    Customers=('customer_id', 'count'),
    Avg_Revenue=('Monetary', 'mean'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean')
).reset_index()
seg_summary['Pct'] = (seg_summary['Customers'] / seg_summary['Customers'].sum() * 100).round(1)

fig = px.treemap(seg_summary, path=['Segment'], values='Customers',
                 color='Avg_Revenue', color_continuous_scale='Blues',
                 title='Customer Segments (Size = Count, Color = Avg Revenue)')
fig.update_layout(width=900, height=500)
fig.show()
print("\nSegment Summary:")
print(seg_summary.sort_values('Customers', ascending=False).to_string(index=False))

In [ ]:
# Segment bar chart
fig, ax = plt.subplots(figsize=(12, 6))
colors_map = {'Champions': '#27AE60', 'Loyal Customers': '#2ECC71', 'Potential Loyalists': '#3498DB',
              'New Customers': '#1ABC9C', 'Need Attention': '#F39C12', 'At Risk': '#E74C3C',
              'Hibernating': '#E67E22', 'Lost Customers': '#C0392B', 'Others': '#95A5A6'}
bar_colors = [colors_map.get(s, '#95A5A6') for s in segment_counts.index]
segment_counts.plot.barh(ax=ax, color=bar_colors, edgecolor='white')
ax.set_title('Customers per Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Customers')
for i, v in enumerate(segment_counts.values):
    ax.text(v + 5, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/rfm_segments.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Export RFM Scores

In [ ]:
rfm.to_csv('../outputs/rfm_scores.csv', index=False)
print(f"Exported {len(rfm):,} customer RFM scores to outputs/rfm_scores.csv")
rfm.head()

## 7. Key RFM Insights

| # | Insight |
|---|---|
| 1 | **Champions** are the most valuable — high R, F, and M |
| 2 | **At Risk** customers have high frequency but haven't purchased recently |
| 3 | **Lost Customers** score low on all three metrics — may need win-back campaigns |
| 4 | **New Customers** purchased recently but only once — nurture them |
| 5 | RFM scores enable **targeted marketing** for each segment |

---
*Proceed to `03_Customer_Segmentation.ipynb`*